# Onafhankelijke E2-schaal vast na epoch 50
## Voortzetting tot epoch 100 · CNN-8 / SVHN · E4M3 → E2M3 → E2M1 · A32 · B32

**Onderzoeksvraag:** wat verandert wanneer we vanaf hetzelfde checkpoint van epoch 50 stoppen met het leren van de absolute E2-schaal en de gewichten verder trainen tot epoch 100?

De run herstelt het **oorspronkelijke epoch-50-checkpoint** en voert **50 nieuwe epochs (51–100)** uit. De eerste 50 epochs worden als bestaande geschiedenis meegenomen. De vijf eerdere freeze-pilotepochs zijn technische context en worden hier niet als nieuwe resultaten overgenomen.

De gewichten en de drie BN-banken leren verder. De E4-schaal volgt de bestaande dynamische schaalberekening; de absolute E2-schaal voor E2M3 en E2M1 blijft vast op haar waarde uit epoch 50.

**Er zijn vooraf geen nieuwe accuracyresultaten ingevuld.** Deze notebook leest alleen beschikbare JSON/CSV-rapporten, controleert hun samenhang en maakt onderscheid tussen nog niet gestart, gedeeltelijk uitgevoerd en voltooid. De officiële testset wordt ook gebruikt voor checkpointselectie; de geselecteerde scores zijn geen onafhankelijke testschatting.


In [ ]:
from pathlib import Path
RUN_DIR = Path('C:\\Users\\renzo\\master-thesis-mx\\mx_runs\\mx_triple_independent_e2_scale_freeze50_reference100\\20261113_140849_905645_joint_check')
SAVE_FIGURES = False


In [ ]:
import importlib.util
import sys
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

package_name = "mx_triple_independent_e2_scale_freeze50_reference100_v1"
starts = [Path(RUN_DIR).resolve()] if RUN_DIR is not None else []
starts += [Path.cwd().resolve()]
candidates = []
for start in starts:
    candidates.append(start / "notebook_support.py")
    for parent in [start, *list(start.parents)[:4]]:
        candidates.append(parent / "software_checks" / package_name / "notebook_support.py")
        candidates.append(parent / package_name / "notebook_support.py")
helper_path = next((path for path in candidates if path.is_file()), None)
if helper_path is None:
    raise FileNotFoundError("Open deze notebook vanuit de resultatenmap of vul RUN_DIR in; notebook_support.py moet beschikbaar zijn.")
spec = importlib.util.spec_from_file_location("mx_independent_freeze50_notebook", helper_path)
nb = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = nb
spec.loader.exec_module(nb)
run = nb.load_run(RUN_DIR)
figures = {}
display(Markdown(nb.status_markdown(run)))
display(nb.diagnostics_table(run))

## 1. Protocol en herkomst

De E2-schaal is **absoluut**. Het bevriezen ervan verschilt van het bevriezen van een relatieve factor: bij die laatste zou E2 nog met E4 meebewegen. Hier blijft de schaalwaarde zelf vast. Het wijzigen van de gewichten kan nog altijd tot andere elementcodes leiden.

In [ ]:
display(nb.protocol_table(run))
display(nb.reference_table(run))
display(Markdown(nb.comparison_markdown(run)))

## 2. Volledige accuracyontwikkeling

Iedere precisie heeft een eigen paneel. De freezevertakking begint bij hetzelfde checkpoint van epoch 50; het grijze gebied is de overgenomen voorgeschiedenis. De verticale lijn markeert de overgang vóór epoch 51. De volledige curves behouden alle getoonde punten binnen de ingezoomde assen.

In [ ]:
figures["accuracy_volledig"] = nb.plot_learning(run)
display(figures["accuracy_volledig"])
plt.close(figures["accuracy_volledig"])

### Detail: de nieuwe vertakking en haar doorleercontrole

Epoch 50 is het gezamenlijke vertrekpunt. Nieuwe punten komen uitsluitend uit daadwerkelijk uitgevoerde epochs 51–100. De historische controle wordt op hetzelfde beschikbare epochvenster getoond; bij een gedeeltelijke run verschijnen geen toekomstige punten.


In [ ]:
figures["accuracy_vertakking"] = nb.plot_final_window(run)
display(figures["accuracy_vertakking"])
plt.close(figures["accuracy_vertakking"])

## 3. Accuracy op hetzelfde eindpunt

**Varianten staan op de verticale as; accuracy op de horizontale as.** E4M3, E2M3 en E2M1 hebben elk een apart paneel. Elk punt is dezelfde cirkel en draagt zijn percentage. Een gedeeltelijke run toont haar werkelijk laatste epoch. Epoch 100 verschijnt pas wanneer de bijbehorende resultaten beschikbaar zijn.


In [ ]:
display(nb.checkpoint_table(run, "last"))
figures["accuracy_eindpunt"] = nb.plot_checkpoints(run, "last")
display(figures["accuracy_eindpunt"])
plt.close(figures["accuracy_eindpunt"])
display(nb.difference_table(run))

## 4. Gemeenschappelijk checkpoint gekozen op de hoogste E4M3-accuracy

Voor elke variant wordt binnen **epochs 1–laatste** de eerste hoogste E4M3-testaccuracy gekozen. De drie precisies worden vervolgens op diezelfde epoch getoond. De gekozen epoch staat bij de variant. Dit checkpoint kan vóór de freeze liggen; dan bevat het nog geen effect van de nieuwe ingreep.

In [ ]:
display(nb.checkpoint_table(run, "best"))
figures["accuracy_beste_E4_geheel"] = nb.plot_checkpoints(run, "best")
display(figures["accuracy_beste_E4_geheel"])
plt.close(figures["accuracy_beste_E4_geheel"])

### Aanvullend: beste E4M3 uitsluitend binnen de nieuwe epochs

Deze selectie gebruikt **epochs 51–laatste**. Zij voorkomt dat een sterk checkpoint uit de gedeelde voorgeschiedenis alle branchverschillen verbergt. Dit is een aanvullende diagnose; de eerder afgesproken globale selectie blijft hierboven staan. Ook hier komen alle drie de modi uit hetzelfde gekozen checkpoint.

In [ ]:
display(nb.checkpoint_table(run, "branch_best"))
figures["accuracy_beste_E4_vertakking"] = nb.plot_checkpoints(run, "branch_best")
display(figures["accuracy_beste_E4_vertakking"])
plt.close(figures["accuracy_beste_E4_vertakking"])

## 5. Bewijs dat de absolute E2-schaal vast blijft

De lezer controleert de continue φ-waarden en de afgeronde absolute E2-exponenten tegen epoch 50. De runner controleert tijdens de nieuwe updates bovendien dat de schaal-Adam-toestand niet verandert en dat φ geen opgeslagen gradient krijgt. Een veranderende geobserveerde verhouding E2/E4 kan wél optreden door beweging van E4.

In [ ]:
display(nb.freeze_evidence_table(run))
figures["freeze_wissels"] = nb.plot_freeze_comparison(run)
display(figures["freeze_wissels"])
plt.close(figures["freeze_wissels"])

### Absolute schaalverdeling en geobserveerde verhouding

Links staan absolute E2-exponenten; rechts staat de achteraf berekende verhouding tot E4. De verhouding is geen geleerd keuzemenu. De oude E2-schaal kan daardoor gelijk blijven terwijl het rechter histogram verschuift.

In [ ]:
display(nb.scale_distribution_table(run, epoch=50) if run.last_epoch is not None else nb.scale_distribution_table(run))
display(nb.scale_distribution_table(run))
figures["schaalverdelingen"] = nb.plot_scale_distributions(run)
display(figures["schaalverdelingen"])
plt.close(figures["schaalverdelingen"])

### Verdeling en gemiddelden gedurende het volledige traject

Epochs 0–50 zijn afkomstig uit de ouder. Vanaf epoch 51 moet de absolute E2-verdeling identiek blijven; de verhouding en E4-bronschaal mogen veranderen.

In [ ]:
figures["schaalevolutie"] = nb.plot_scale_evolution(run)
display(figures["schaalevolutie"])
plt.close(figures["schaalevolutie"])
display(nb.scale_layer_table(run))
figures["schalen_per_laag"] = nb.plot_scale_by_layer(run)
display(figures["schalen_per_laag"])
plt.close(figures["schalen_per_laag"])

### Wissels, gradients en grensprojecties

Een wisselgebeurtenis kan herhaald dezelfde groep betreffen. Dat aantal is niet hetzelfde als het aantal groepen met een andere eindwaarde. Vanaf epoch 51 zijn de schaalgradientkolommen in het rapport nul als conventie voor **uitgeschakeld**; `φ.grad` is afwezig. Dit is geen gemeten nulgradient en geen bewijs van convergentie.

In [ ]:
figures["schaalveranderingen_volledig"] = nb.plot_scale_changes(run)
display(figures["schaalveranderingen_volledig"])
plt.close(figures["schaalveranderingen_volledig"])
figures["schaaloptimalisatie"] = nb.plot_scale_gradients(run)
display(figures["schaaloptimalisatie"])
plt.close(figures["schaaloptimalisatie"])

## 6. Opslag en bijkomende kosten

De freeze verandert de compacte representatie niet: er blijven één E4-schaalbyte en één onafhankelijke E2-schaalbyte per groep nodig. Voor 4.080 groepen is dat **8.160 bytes schaalmetadata**: +4.080 bytes tegenover vaste ×64 en +2.040 bytes tegenover vier bits voor tien relatieve keuzes.

De tabel onderscheidt compacte codes en schaalmetadata van de nominale trainingshulppayload. In de freezecheckpoint blijven FP32-proxies en twee Adam-momenten bewaard voor herkomst en hervatten; hun opgeslagen gradients zijn afwezig. Dat betekent niet dat tijdelijke berekeningen of alle frameworkoverhead verdwijnen. Randlagen, BN, bestandheaders en allocatoroverhead staan niet in de berekende MX-payload.

**Opslag bewijst geen DRAM-transfer-, latency- of energiewinst.** Die vereisen een afzonderlijke meting met vastgelegde caching, packing en hardware. Er blijven drie precisiepasses per minibatch.

In [ ]:
display(nb.storage_cost_table(run))
figures["opslagkosten"] = nb.plot_storage_cost(run)
display(figures["opslagkosten"])
plt.close(figures["opslagkosten"])
display(nb.export_table(run))

## 7. Tijd van alleen de nieuw uitgevoerde epochs

De eerste 50 epochs worden niet opnieuw uitgevoerd en tellen niet mee als nieuwe looptijd van deze run. Tot epoch 100 is bereikt, wordt een eventuele resterende duur als schatting gemarkeerd. Na voltooiing staat hier de **gemeten tijd van alle 50 nieuwe epochs**; de historische tijd van epochs 1–50 blijft afzonderlijk herkenbaar. De optelsom beschrijft een traject met hergebruikte voorgeschiedenis, geen training van 100 nieuwe epochs.


In [ ]:
display(Markdown(nb.timing_markdown(run)))
figures["looptijd"] = nb.plot_timing(run)
display(figures["looptijd"])
plt.close(figures["looptijd"])

## 8. Softwarecontrole en interpretatie

Een technische PASS bevestigt correcte uitvoering, geen accuracywinst. Beoordeel dezelfde eindepoch, de gemeenschappelijke E4-selectie, het nieuwe venster vanaf epoch 51, de absolute E2-wissels en de accuracystabiliteit.

De vergelijking betreft één vertakking van één seed. Opeenvolgende epochs zijn geen onafhankelijke herhalingen. De GPU-backend garandeert geen bit-exacte herhaling; deze nieuwe start hoeft de eerdere vijfepochpilot daarom niet exact te reproduceren. Een resultaat op epoch 100 kan alleen worden geconcludeerd uit de werkelijk voltooide nieuwe run.


In [ ]:
display(nb.audit_table(run))
display(nb.diagnostics_table(run))

## 9. Optioneel figuren bewaren

Zet `SAVE_FIGURES = True` in de parametercel om afgeleide PNG- en PDF-figuren naar een nieuwe map te schrijven. Ruwe resultaten en bestaande figuren worden niet overschreven. De standaardinstelling bewaart niets.

In [ ]:
if SAVE_FIGURES:
    figure_directory = nb.save_figures(figures, run)
    display(Markdown(f"Figuren opgeslagen in: `{figure_directory}`"))
else:
    display(Markdown("Figuren zijn alleen weergegeven; `SAVE_FIGURES = False`."))